# AGEMR：ACAT、训练/验证 panel-PGS×age、block variance-component score

本 notebook 比较三种不同的集合问题：

- **PGS×age：** 按训练集估计的主效应权重构成候选面板 PGS，在验证集检验一个 score×age 项。它是方向一致的 burden 路线。
- **Variance-component 路线：** 在验证集对一组 G×age 项做 kernel score 检验，允许正负效应并存；每个块的所有 SNP 主效应均进入 null model。
- **ACAT：** 聚合单 SNP 的交互 P，也可聚合块级检验 P。ACAT 是尾部近似的 P-value combination，不是方差分量模型，也不估计解释方差。

**已有结果部分只读文件；新分析部分读取现有 workspace PGEN，不下载 controlled data。** 新文件留在同一 workspace 的新目录。PGS 是基于当前 1,902 个候选位点的 panel score，不称为完整 genome-wide PRS。

**v2 修复 PLINK2 定位：** 优先读取旧 B1 与 pooled-main 命令中实际使用的程序路径，再检查常见软件位置。逐项记录 missing、PermissionError、动态库/启动错误。程序已丢失时，默认仅从公开 Bioconda 安装固定版本到 driver-local 临时软件目录；不更改 Python/Spark 环境，也不把程序安装到 workspace 数据挂载。设 `INSTALL_PLINK_IF_MISSING=False` 可关闭软件安装。A 节不需要 PLINK。

用户已选择固定训练/验证分割。每个 IID 用固定 hash 分配 50%/50%；三个表型共享同一 IID 分配。权重和主效应 P 分组只在训练集计算，避免用验证集结局训练 PGS。个体分割不自动排除跨集亲缘关系；如有无关个体 keep，可在配置中指定。默认继承现有 cohort，是探索性验证。

**运行顺序：** 先执行 A（已有 full-sample ACAT）。B 为新训练/验证分析，会运行 PLINK 和多个 kernel，明显重于之前只读汇总；默认运行。8 GB driver 不构造 n×n kernel，不把全部 1,902 个 SNP 同时放进一个密集 null model：按预定染色体/位点块检验，再合并块 P。结果称为 **block-VC–ACAT**，不是完整联合 GRM/REML 或全基因组交互方差估计。

In [ ]:
from pathlib import Path
import json,hashlib,uuid,subprocess,shutil,warnings,gc,tempfile,os
from datetime import datetime,timezone
import numpy as np
import pandas as pd
from scipy import stats,linalg
from IPython.display import display

LINEAR_RUN=None
B1_ROOT=Path.home()/"workspace/agemr-data/final_cohort_v1/B1_age_varying"
PLINK_EXECUTABLE=None
INSTALL_PLINK_IF_MISSING=True
PLINK_ENV=Path(tempfile.gettempdir())/("agemr-tools-"+str(os.getuid()))/"plink2_2.0.0a.7.8"
RUN_TRAIN_TEST=True  # False: only section A, no genotype access or PLINK calls.
UNRELATED_KEEP=None  # Optional IID-only / FID=0 keep file; no new kinship inference is assumed.
TRAITS=("BMI","SBP","DBP")
GROUPS=[("All",None,"W_All"),("P_main < 1e-3",1e-3,"W_1e3"),
        ("P_main < 1e-5",1e-5,"W_1e5"),("P_main < 5e-8",5e-8,"W_5e8")]
KEY=["#CHROM","POS","ID","REF","ALT"]
SPLIT_SEED=20261028
THREADS=2
MEMORY_MB=1024
VC_MIN_TRAIN_MAC=20
VC_MIN_TEST_MAC=10
VC_BLOCK_SNPS=128  # Prespecified blocks; split only by chromosome and position, not test results.
VC_N_SIM=49999
VC_SEED=20261028
MIN_TEST_N=100
PRIMARY_FAMILY=36  # 3 traits x 4 training-defined sets x (PGS, heldout ACAT, block-VC–ACAT)

def require(ok,message):
    if not ok:raise RuntimeError(message)

def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda:f.read(1024*1024),b""):h.update(b)
    return h.hexdigest()

if LINEAR_RUN is None:
    candidates=[]
    for marker in (B1_ROOT/"B1_linear_age").glob("*/RUN_COMPLETE.json"):
        m=json.loads(marker.read_text())
        if m.get("status")=="complete" and all((marker.parent/"main_effect_stratified_proportions"/f"{t}_pooled_main.tsv").is_file() for t in TRAITS):
            candidates.append((m.get("finished_utc",""),marker.parent))
    require(bool(candidates),"No completed linear+pooled-main run found; set LINEAR_RUN.")
    LINEAR_RUN=max(candidates,key=lambda z:z[0])[1]
else:LINEAR_RUN=Path(LINEAR_RUN).expanduser()
completion=json.loads((LINEAR_RUN/"RUN_COMPLETE.json").read_text())
require(completion.get("status")=="complete","Incomplete selected linear run")
M=int(completion["panel_variants"])
stamp=datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"_"+uuid.uuid4().hex[:6]
OUT=LINEAR_RUN/"PGS_age_VC_ACAT"/stamp
for folder in (OUT,OUT/"A_existing_ACAT",OUT/"B_holdout",OUT/"qc"):
    folder.mkdir(parents=True,exist_ok=True)
print("Using existing results:",LINEAR_RUN)
print("New outputs:",OUT)

sources=[]
def read_result(path,unique=True):
    x=pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False)
    require(set(KEY).issubset(x),f"Unexpected schema: {path}")
    x["#CHROM"]=x["#CHROM"].str.replace(r"^chr","",regex=True)
    x["POS"]=pd.to_numeric(x.POS,errors="raise").astype(int)
    if unique:require(not x.duplicated(KEY).any(),f"Duplicate variant keys: {path}")
    for col in ("P","BETA","SE","OBS_CT","A1_FREQ","T_STAT","T_OR_F_STAT"):
        if col in x:x[col]=pd.to_numeric(x[col],errors="coerce")
    sources.append({"path":str(path),"sha256":sha(path)})
    return x

def valid_p(x):return np.isfinite(x)&(x>=0)&(x<=1)
def truth(x):return x.astype(str).str.lower().isin(["true","1"])

def acat(p,weights=None):
    p=np.asarray(p,dtype=float)
    require(valid_p(p).all(),"ACAT needs finite P in [0,1]; do not silently remove invalid inputs")
    if len(p)==0:return {"P_ACAT":np.nan,"N":0,"N_zero_P":0,"status":"empty"}
    w=np.ones(len(p)) if weights is None else np.asarray(weights,dtype=float)
    require(w.shape==p.shape and np.isfinite(w).all() and (w>=0).all() and w.sum()>0,"Invalid ACAT weights")
    keep=w>0;p=p[keep];w=w[keep]/w[keep].sum()
    nz=int((p==0).sum());no=int((p==1).sum())
    if (p==1).all():return {"P_ACAT":1.0,"N":len(p),"N_zero_P":nz,"N_one_P":no,"status":"all_P_one"}
    q=np.clip(p,1e-300,1-1e-15)
    values=np.empty(len(q))
    small=q<1e-8;large=q>1-1e-8;middle=~(small|large)
    values[small]=1/(np.pi*q[small])
    values[large]=-1/(np.pi*(1-q[large]))
    values[middle]=np.tan(np.pi*(.5-q[middle]))
    T=float(np.sum(w.astype(np.longdouble)*values.astype(np.longdouble),dtype=np.longdouble))
    if T>0:P=np.arctan(1/T)/np.pi
    elif T<0:P=1-np.arctan(-1/T)/np.pi
    else:P=.5
    positive=np.maximum(w*values,0)
    dominance=float(positive.max()/positive.sum()) if positive.sum()>0 else np.nan
    return {"P_ACAT":float(np.clip(P,0,1)),"N":len(p),"T_ACAT":T,"N_zero_P":nz,"N_one_P":no,
            "min_P":float(p.min()),"minP_Bonf":float(min(len(p)*p.min(),1)),
            "largest_positive_Cauchy_contribution_fraction":dominance,
            "status":"underflow_P_clipped_reference" if nz else "ACAT_tail_approximation"}

## A. 直接对已有 full-sample 1-df / 旧 4-df P 做 ACAT

这里沿用原来的 pooled-main 分层，因此是既有数据的探索性聚合；不与下节训练定义的子集混为同一分析。先统一三份结果的共同有效 SNP 和 OBS_CT，再各自做 equal-weight ACAT。四组×三表型×两模型共 24 个计划检验，报告固定 24 倍校正。

同时报告最小单 SNP P、min-P Bonferroni、最大 Cauchy 正贡献占比，以及去掉最小 P 后的 ACAT，帮助识别是否只由一个位点驱动。**ACAT 对很小的 P 很敏感，不是专门的 dense weak-effect variance test**。相关性下是尾部近似，不等于任意依赖、任何 P 范围都精确；坏的输入 P 不会被 ACAT 修复。P=0 时明确标记数值下溢并以 1e-300 作计算参考，不把它当精确 P。

参考：[ACAT 原始论文](https://pmc.ncbi.nlm.nih.gov/articles/PMC6407498/)、[Cauchy combination 理论](https://pmc.ncbi.nlm.nih.gov/articles/PMC7531765/)。

In [ ]:
existing=[]
for trait in TRAITS:
    one=read_result(LINEAR_RUN/f"{trait}_linear_age_all.tsv")
    main=read_result(LINEAR_RUN/"main_effect_stratified_proportions"/f"{trait}_pooled_main.tsv")
    snapshot=LINEAR_RUN/f"{trait}_linear_vs_old_4df.tsv"
    if snapshot.is_file():
        s=read_result(snapshot)
        old=s.loc[s.TEST_4df.eq("USER_4DF"),KEY].copy()
        old["P_4df"]=pd.to_numeric(s.loc[old.index,"P_4df"],errors="coerce")
        old["N_4df"]=pd.to_numeric(s.loc[old.index,"OBS_CT_4df"],errors="coerce")
        old["ok_4df"]=s.loc[old.index,"ERRCODE_4df"].eq(".")
        if "VALID_4df" in s:old["ok_4df"] &= truth(s.loc[old.index,"VALID_4df"])
        if "old_log_parameter_status" in s:old["ok_4df"] &= s.loc[old.index,"old_log_parameter_status"].ne("parameters_mismatch_or_unreadable")
    else:
        s=read_result(LINEAR_RUN.parent.parent/f"{trait}_age_varying_joint_tests.tsv")
        require(s.TEST.eq("USER_4DF").all(),"Old results must be USER_4DF")
        old=s[KEY].copy();old["P_4df"]=s.P;old["N_4df"]=s.OBS_CT;old["ok_4df"]=s.ERRCODE.eq(".")
    x=one[KEY].copy();x["P_1df"]=one.P;x["N_1df"]=one.OBS_CT
    x["ok_1df"]=truth(one.VALID)&one.ERRCODE.eq(".")
    m=main[KEY].copy();m["P_main"]=main.P;m["N_main"]=main.OBS_CT;m["ok_main"]=main.ERRCODE.eq(".")
    x=x.merge(m,on=KEY,how="left",validate="one_to_one").merge(old,on=KEY,how="left",validate="one_to_one")
    x["eligible"]=(x.ok_1df & truth(x.ok_main) & truth(x.ok_4df) & valid_p(x.P_main)&valid_p(x.P_1df)&valid_p(x.P_4df)&
                   x.N_main.eq(x.N_1df)&x.N_main.eq(x.N_4df))
    x.to_csv(OUT/"A_existing_ACAT"/f"{trait}_matched_inputs.tsv",sep="\t",index=False)
    for label,cut,_ in GROUPS:
        selected=pd.Series(True,index=x.index) if cut is None else x.P_main.lt(cut)
        g=x.loc[selected & x.eligible]
        for model in ("1df","4df"):
            p=g[f"P_{model}"].to_numpy(dtype=float)
            r=acat(p)
            drop=acat(np.delete(p,np.argmin(p)))["P_ACAT"] if len(p)>1 else np.nan
            existing.append({"trait":trait,"group":label,"model":model,"N_selected_by_main_P":int(selected.sum()),
                             **r,"P_ACAT_drop_minP":drop})
existing=pd.DataFrame(existing)
existing["P_ACAT_Bonf24"]=np.minimum(existing.P_ACAT*24,1)
existing.to_csv(OUT/"A_existing_ACAT/existing_1df_4df_ACAT.tsv",sep="\t",index=False)
display(existing)

## B. 固定训练/验证：PGS×age 与 block-VC–ACAT

训练集按保存的完整 age spline、sex、QPC 调整跑 pooled GWAS；从训练 P 形成四个集合及训练 beta 权重。验证集做：

1. **panel-PGS×AGE10**：`Y ~ saved nuisance + standardized_PGS + standardized_PGS×AGE10`。PGS 的 SNP 分数先按染色体 **SUM** 相加，不把 chromosome-average 相加；缺失 genotype 使用训练集 allele frequency 均值填补。主 P 使用 HC3 robust SE 的大样本正态参考，普通 OLS t P 作辅助。报告单位为“每验证集 PGS SD、每 10 年”的交互效应。
2. **Heldout single-SNP ACAT**：训练集决定集合，验证集对同一 1-df SNP×age 模型产生普通 OLS P，按组 ACAT。这些单 SNP P 沿用原分析的常规标准误；不同于本节 PGS 的 HC3 与 kernel 的 robust score 校准，需分别审查其误差假设。
3. **block-VC–ACAT**：将 SNP 按染色体/POS 预先划为最多 128 个位点的块。每块、每组在验证集拟合 null `Y ~ nuisance + all SNP main effects in this block/set`；用 residualized G×AGE10 kernel 的 Q=||Hᵀr||²。主块 P 用 HC3 score covariance 的特征值与 Gaussian multiplier/weighted-chi-square Monte Carlo 参考；它是 heteroskedasticity-robust 的渐近 score 近似。另存 homoskedastic Gaussian 模型下、以 residual variance 估计缩放的有限样本 ratio-null Monte Carlo 作敏感性。块 P 再以 ACAT 合并，权重按块内位点数，保持每 SNP 的名义块权重一致。

不形成 n×n 矩阵，按块处理 genotype。VC 另要求训练 MAC≥20、验证 genotype MAC≥10，并排除无可识别交互方向的列，避免极少携带者导致高杠杆；PGS 与单 SNP ACAT 保留各自的有效训练集合，因此有效 SNP 数未必完全相同。block-VC–ACAT 与“把全部 1,902 SNP 同时拟合的单一 kernel/GRM”不同；PGS 只调一个 score main effect，VC null 调块内各 SNP main effect，两个模型也不完全相同。

Monte Carlo 默认 49,999 次，最小报告 P 为 2×10⁻⁵；与连续尾部 ACAT 合并时可能限制分辨率，结果会记录触底块数。若信号触底，增大 VC_N_SIM 后重新核查，不把该数值当精确极小 P。只有一个非零 eigenvalue 时用解析 χ²₁ 尾部，无 MC 截断。

本节三个方法×四集合×三表型，固定 **36 项 Bonferroni**。空集合/常数 PGS/无法估计的块明确记录，不静默换一个更有利的集合。A 节与 B 节的样本和分组不同，结果不能直接当相同试验的重复。所有模型仍需满足样本/协变量/误差及相关性假设。

In [ ]:
def run_cmd(command,prefix):
    prefix=Path(prefix);prefix.parent.mkdir(parents=True,exist_ok=True)
    Path(str(prefix)+".command.json").write_text(json.dumps(command,indent=2))
    with Path(str(prefix)+".stdout.txt").open("w") as out,Path(str(prefix)+".stderr.txt").open("w") as err:
        result=subprocess.run(command,stdout=out,stderr=err)
    if result.returncode:
        detail=Path(str(prefix)+".stderr.txt").read_text(errors="replace")[-2000:]
        raise RuntimeError(f"PLINK failed ({result.returncode}): {prefix}\n{detail}")

def iid_table(path):
    x=pd.read_csv(path,sep="\t",dtype=str,keep_default_na=False).rename(columns={"#IID":"IID","#FID":"FID"})
    require("IID" in x and x.IID.is_unique,"IID missing or duplicate")
    if "FID" in x:
        require(x.FID.eq("0").all(),"This workflow requires IID-only/FID=0 files")
        x=x.drop(columns="FID")
    return x.set_index("IID")

def is_train(iid):
    h=hashlib.sha256(f"{SPLIT_SEED}|{iid}".encode()).digest()
    return int.from_bytes(h[:8],"big") < (1<<63)

def ols_pgs_hc3(y,C,pgs,age):
    sd=np.std(pgs,ddof=1)
    if not np.isfinite(sd) or sd<=1e-12:return {"status":"constant_or_empty_score","N":len(y)}
    score=(pgs-np.mean(pgs))/sd
    X=np.column_stack([np.ones(len(y)),C,score,score*age])
    beta,_,rank,_=np.linalg.lstsq(X,y,rcond=None)
    if rank!=X.shape[1]:return {"status":"rank_deficient","N":len(y)}
    residual=y-X@beta;df=len(y)-rank
    inv=np.linalg.inv(X.T@X)
    leverage=np.einsum('ij,jk,ik->i',X,inv,X)
    require(np.max(leverage)<1-1e-8,"HC3 leverage too high")
    adjusted=residual/(1-leverage)
    V=inv@((X*adjusted[:,None]).T@(X*adjusted[:,None]))@inv
    se=float(np.sqrt(V[-1,-1]));classic=np.sqrt((residual@residual/df)*inv[-1,-1])
    z=beta[-1]/se
    return {"status":"ok","N":len(y),"BETA_PGSxAGE10":beta[-1],"SE_HC3":se,
            "P":float(2*stats.norm.sf(abs(z))),"CI95_low":beta[-1]-1.959964*se,"CI95_high":beta[-1]+1.959964*se,
            "PGS_SD_before_standardization":sd,"SE_classic":classic,
            "P_classic":float(2*stats.t.sf(abs(beta[-1]/classic),df)),"PGS_main_BETA":beta[-2]}

def mixture_p(Q,eigenvalues,seed,df_resid=None,B=VC_N_SIM):
    lam=np.asarray(eigenvalues,dtype=float)
    require(np.isfinite(lam).all(),"Nonfinite kernel eigenvalues")
    if not len(lam) or lam.max()<=0:return 1.0,"zero_kernel",0
    require(lam.min()>=-1e-7*lam.max(),"Substantially negative kernel eigenvalues")
    lam=lam[lam>lam.max()*1e-10]
    if len(lam)==0:return 1.0,"zero_kernel",0
    if len(lam)==1 and df_resid is None:return float(stats.chi2.sf(Q/lam[0],1)),"analytic_chi2_1",0
    if len(lam)==1 and df_resid is not None:
        # Q/(lambda*df) ~ Beta(1/2,(df-1)/2), exact under homoskedastic Gaussian residuals.
        return float(stats.beta.sf(np.clip(Q/(lam[0]*df_resid),0,1),.5,(df_resid-1)/2)),"analytic_Gaussian_ratio_rank1",0
    rng=np.random.default_rng(seed);exceed=0;done=0
    while done<B:
        batch=min(512,B-done)
        zz=rng.chisquare(1,size=(batch,len(lam)))
        q=zz@lam
        if df_resid is not None:
            require(df_resid>len(lam),"Insufficient residual degrees of freedom")
            q=q*df_resid/(zz.sum(axis=1)+rng.chisquare(df_resid-len(lam),size=batch))
        exceed+=int(np.sum(q>=Q));done+=batch
    return (exceed+1)/(B+1),"Monte_Carlo_ratio" if df_resid is not None else "Gaussian_multiplier_eigen_MC",B

def kernel_score(y,C,G,age,seed):
    require(len(y)>MIN_TEST_N,"Test sample too small")
    # G is standardized, training-mean-imputed ALT dosage.
    variable=G.std(axis=0)>1e-10
    G=G[:,variable]
    if G.shape[1]==0:return {"status":"no_variable_SNP","N":len(y),"N_SNP":0}
    W=np.column_stack([np.ones(len(y)),C,G])
    Qw,R,piv=linalg.qr(W,mode="economic",pivoting=True,check_finite=False)
    diagonal=np.abs(np.diag(R));rank=int((diagonal>max(W.shape)*np.finfo(float).eps*diagonal.max()).sum())
    Qw=Qw[:,:rank];del W
    r=y-Qw@(Qw.T@y)
    H=G*age[:,None]
    original_norm=np.linalg.norm(H,axis=0)
    H=H-Qw@(Qw.T@H)
    identifiable=np.linalg.norm(H,axis=0)>1e-9*np.maximum(original_norm,1.0)
    H=H[:,identifiable]
    if not H.shape[1]:return {"status":"no_identifiable_interaction","N":len(y),"N_SNP":0}
    nH=H.shape[1]
    U=H.T@r;Qstat=float(U@U)
    leverage=np.sum(Qw*Qw,axis=1);df=len(y)-rank
    if df<=nH or np.max(leverage)>=1-1e-8:
        return {"status":"insufficient_df_or_excessive_leverage","N":len(y),"N_SNP":0}
    V=np.zeros((nH,nH))
    for start in range(0,len(y),4096):
        stop=min(start+4096,len(y));Bmat=H[start:stop]*(r[start:stop]/(1-leverage[start:stop]))[:,None]
        V+=Bmat.T@Bmat
    eig=linalg.eigvalsh(V,check_finite=False)
    p,method,nsim=mixture_p(Qstat,eig,seed)
    sigma2=float(r@r/df)
    eigK=linalg.eigvalsh(H.T@H,check_finite=False)
    pclassic,methodclassic,_=mixture_p(Qstat/sigma2,eigK,seed+1,df_resid=df)
    return {"status":"ok","N":len(y),"N_SNP":nH,"N_null_main_SNP":G.shape[1],"null_rank":rank,"DF_RESID":df,
            "Q_score":Qstat,"P":p,"calibration":method,"N_sim":nsim,
            "MC_at_floor":bool(nsim and p<=1/(nsim+1)),
            "P_homoskedastic_Gaussian":pclassic,"Gaussian_method":methodclassic,
            "positive_eigenvalues":int((eig>max(eig.max(),0)*1e-10).sum())}

# Embedded in each delivered notebook; no external Python file is needed.
def resolve_existing_or_install_plink(preferred, linear_run, log_dir, install_if_missing, env_prefix):
    import os, re, sys, tempfile
    log_dir = Path(log_dir); log_dir.mkdir(parents=True, exist_ok=True)
    env_prefix = Path(env_prefix).expanduser()
    audit, candidates, seen = [], [], set()

    def add(value, origin):
        if value is not None and str(value).strip():
            value = os.path.expandvars(str(value))
            found = shutil.which(value) if not Path(value).is_absolute() else None
            path = str(Path(found or value).expanduser().absolute())
            if path not in seen:
                seen.add(path); candidates.append((path, origin))

    def save():
        (log_dir/"plink_discovery.json").write_text(json.dumps(audit, indent=2))

    add(preferred, "explicit configuration")
    # Read program paths only; never replay a historical command or use shell=True.
    run = Path(linear_run)
    old_audit = run/"qc/plink_discovery.json"
    if old_audit.is_file():
        try:
            old = json.loads(old_audit.read_text())
            if isinstance(old, list):
                for r in old:
                    if isinstance(r, dict) and r.get("status") == "usable":
                        add(r.get("path"), "B1 successful discovery record")
        except (OSError, ValueError): pass
    historical = list(run.glob("*_results/*.command.json"))
    historical += list((run/"main_effect_stratified_proportions/pooled_main").glob("*/*.successful_signature.json"))
    for path in sorted(historical)[:200]:
        try:
            entry = json.loads(path.read_text())
            command = entry.get("command") if isinstance(entry, dict) else entry
            if isinstance(command,list) and command and isinstance(command[0],str):
                if Path(command[0]).name.lower() in ("plink2", "plink2.exe"):
                    add(command[0], "previous successful/run command: "+str(path))
        except (OSError, ValueError): continue
    for p in [shutil.which("plink2"), env_prefix/"bin/plink2",
              Path.home()/"plink2_env/bin/plink2", Path(sys.prefix)/"bin/plink2",
              Path.home()/".local/bin/plink2", "/home/dataproc/plink2_env/bin/plink2",
              "/home/jupyter/plink2_env/bin/plink2", "/opt/conda/default/bin/plink2",
              "/opt/conda/bin/plink2", "/usr/local/bin/plink2", "/usr/bin/plink2"]:
        add(p, "known software location")
    if env_prefix.parent.is_dir():
        for p in sorted(env_prefix.parent.glob("plink2_*/bin/plink2"))[:50]:
            add(p, "cached driver-local software")

    def probe(path, origin):
        r = {"path":str(path), "origin":origin}
        try:
            if not Path(path).is_file(): r["status"] = "missing"
            else:
                r["executable_mode"] = os.access(path,os.X_OK)
                v = subprocess.run([str(path),"--version"],capture_output=True,text=True,timeout=30)
                r["returncode"] = v.returncode
                r["version_or_error"] = (v.stdout+v.stderr).strip()[-2500:]
                r["status"] = "usable" if v.returncode == 0 and re.search(r"PLINK\s+v?2\.",r["version_or_error"],re.I) else "startup_failed"
        except (OSError, subprocess.TimeoutExpired) as e:
            r.update(status="execution_failed", error=str(e), errno=getattr(e,"errno",None))
        audit.append(r); save()
        if r["status"] == "usable": return str(path)
        return None

    def finish(path):
        print("Using PLINK2:",path,flush=True)
        print(audit[-1].get("version_or_error",""),flush=True)
        (log_dir/"plink_selected.json").write_text(json.dumps(audit[-1],indent=2))
        return path

    for path,origin in candidates:
        found = probe(path,origin)
        if found: return finish(found)
    important = [r for r in audit if r["status"] != "missing"]
    print("PLINK2 search diagnostics:",flush=True)
    print(pd.DataFrame(important or audit).to_string(index=False),flush=True)
    if not install_if_missing:
        raise RuntimeError(f"No runnable PLINK2; diagnostics: {log_dir/'plink_discovery.json'}. Set PLINK_EXECUTABLE or INSTALL_PLINK_IF_MISSING=True.")

    manager = None
    for p in [shutil.which("mamba"), shutil.which("conda"),
              "/opt/conda/default/bin/mamba", "/opt/conda/default/bin/conda", "/opt/conda/bin/conda"]:
        if p and Path(p).is_file() and os.access(p,os.X_OK): manager = str(p); break
    if manager is None:
        raise RuntimeError(f"No runnable PLINK2 and no mamba/conda available. Inspect {log_dir/'plink_discovery.json'}; set PLINK_EXECUTABLE to the working binary.")
    # Do not overwrite a broken prefix or mutate the Spark/Python environment.
    destination = env_prefix
    if destination.exists(): destination = destination.with_name(destination.name+"_"+uuid.uuid4().hex[:8])
    destination.parent.mkdir(parents=True,exist_ok=True)
    command = [manager,"create","-y","-p",str(destination),"--override-channels",
               "-c","conda-forge","-c","bioconda","plink2=2.0.0a.7.8"]
    (log_dir/"plink_install.command.json").write_text(json.dumps(command,indent=2))
    print("Installing public PLINK2 software only into driver-local:",destination,flush=True)
    print("Genotype/phenotype files remain in the workspace; active Python/Spark environment is unchanged.",flush=True)
    with (log_dir/"plink_install.log").open("w") as log:
        proc = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in proc.stdout:
                print(line,end="",flush=True); log.write(line); log.flush()
            returncode = proc.wait()
        except BaseException:
            proc.terminate()
            try: proc.wait(timeout=10)
            except subprocess.TimeoutExpired: proc.kill(); proc.wait()
            raise
    if returncode:
        raise RuntimeError(f"Public software installation failed ({returncode}). See {log_dir/'plink_install.log'}; no genotype analysis has started.")
    found = probe(destination/"bin/plink2","new public software installation")
    if not found:
        raise RuntimeError("PLINK2 was installed but cannot start: "+json.dumps(audit[-1])+f". See {log_dir/'plink_discovery.json'}")
    return finish(found)


In [ ]:
if RUN_TRAIN_TEST:
    PLINK=resolve_existing_or_install_plink(PLINK_EXECUTABLE,LINEAR_RUN,OUT/"qc",INSTALL_PLINK_IF_MISSING,PLINK_ENV)
    records=json.loads((LINEAR_RUN/"qc/input_files.json").read_text())
    pfiles={int(Path(r["path"]).stem.replace("BMI_chr","")):Path(r["path"]).with_suffix("") for r in records if Path(r["path"]).suffix==".pgen"}
    for record in records:
        path=Path(record["path"])
        if path.suffix in (".pgen",".pvar",".psam"):
            st=path.stat()
            require(st.st_size==record["bytes"] and st.st_mtime_ns==record["mtime_ns"],f"Genotype file changed since linear run: {path}")
    panel=pd.read_csv(LINEAR_RUN/"qc/variant_panel.tsv",sep="\t",dtype={"#CHROM":str})
    panel["#CHROM"]=panel["#CHROM"].astype(str).str.replace(r"^chr","",regex=True)
    panel=panel.sort_values(["#CHROM","POS","ID"]).reset_index(drop=True)
    panel["block_number"]=panel.groupby("#CHROM").cumcount()//VC_BLOCK_SNPS
    panel["block"]=panel["#CHROM"]+"_"+panel.block_number.astype(str)
    panel.to_csv(OUT/"qc/fixed_variant_blocks.tsv",sep="\t",index=False)
    unrelated=None
    if UNRELATED_KEEP is not None:
        unrelated=set(iid_table(Path(UNRELATED_KEEP)).index)
    holdout_rows=[];block_rows=[];split_rows=[]
    for trait in TRAITS:
        print("\nTRAIN/TEST:",trait,flush=True)
        d=OUT/"B_holdout"/trait;d.mkdir(exist_ok=True)
        ph_path=LINEAR_RUN/"inputs"/f"{trait}.pheno.tsv";cv_path=LINEAR_RUN/"inputs"/f"{trait}.covar.tsv"
        saved_meta=json.loads((LINEAR_RUN/"inputs"/f"{trait}.meta.json").read_text())
        require(sha(ph_path)==saved_meta["pheno_sha256"] and sha(cv_path)==saved_meta["covar_sha256"],"Saved linear inputs changed")
        ph=iid_table(ph_path)[[trait]].apply(pd.to_numeric,errors="raise").astype(float)
        cv=iid_table(cv_path).loc[ph.index].apply(pd.to_numeric,errors="raise").astype(float)
        if unrelated is not None:ph=ph.loc[ph.index.isin(unrelated)];cv=cv.loc[ph.index]
        require(np.isfinite(ph.to_numpy()).all() and np.isfinite(cv.to_numpy()).all(),"Missing analysis input")
        train_mask=np.array([is_train(i) for i in ph.index]);train=ph.index[train_mask];test=ph.index[~train_mask]
        require(len(train)>=MIN_TEST_N and len(test)>=MIN_TEST_N,"Split samples too small")
        # Remove only covariates constant in either split, keeping AGE10. This avoids PLINK's automatic parameter renumbering.
        drop=[c for c in cv if cv.loc[train,c].std()==0 or cv.loc[test,c].std()==0]
        require("AGE10" not in drop,"No age variation in split")
        cv=cv.drop(columns=drop)
        for ids in [train,test]:require(np.linalg.matrix_rank(np.column_stack([np.ones(len(ids)),cv.loc[ids]]))==cv.shape[1]+1,"Rank-deficient nuisance after split")
        ph.to_csv(d/"pheno.tsv",sep="\t",index_label="#IID",float_format="%.17g")
        cv.to_csv(d/"covar.tsv",sep="\t",index_label="#IID",float_format="%.17g")
        for name,ids in [("train",train),("test",test)]:pd.DataFrame({"#IID":ids}).to_csv(d/f"{name}.keep.tsv",sep="\t",index=False)
        split_rows.append({"trait":trait,"N_train":len(train),"N_test":len(test),"seed":SPLIT_SEED,"dropped_constant_covariates":",".join(drop)})
        train_parts=[];heldout_parts=[];score_total=pd.DataFrame(0.0,index=test,columns=[z[2] for z in GROUPS])
        train_freq={}
        for chrom in completion["chromosomes"]:
            print(f"{trait}: training weights / heldout interaction / PGS chr{chrom}",flush=True)
            prefix=d/f"train_chr{chrom}"
            base=[PLINK,"--pfile",str(pfiles[chrom]),"--pheno",str(d/"pheno.tsv"),"--pheno-name",trait,
                  "--covar",str(d/"covar.tsv"),"--no-input-missing-phenotype","--threads",str(THREADS),"--memory",str(MEMORY_MB)]
            run_cmd(base+["--keep",str(d/"train.keep.tsv"),"--glm","hide-covar","--freq","--out",str(prefix)],prefix)
            tr=read_result(Path(str(prefix)+f".{trait}.glm.linear"))
            require(tr.TEST.eq("ADD").all(),"Training GWAS must be pooled ADD")
            train_parts.append(tr)
            freq=pd.read_csv(Path(str(prefix)+".afreq"),sep="\t",dtype=str)
            require("ALT_FREQS" in freq,"Unexpected frequency schema")
            freq["ALT_FREQS"]=pd.to_numeric(freq.ALT_FREQS,errors="coerce")
            train_freq[chrom]=freq.set_index("ID")
            valid=tr.ERRCODE.eq(".")&valid_p(tr.P)&np.isfinite(tr.BETA)&np.isfinite(tr.SE)&tr.SE.gt(0)
            weights=tr.loc[valid,["ID","A1","BETA","P"]].copy()
            for label,cut,score_col in GROUPS:weights[score_col]=weights.BETA*np.where(True if cut is None else weights.P.lt(cut),1,0)
            weight_path=d/f"weights_chr{chrom}.tsv"
            weights[["ID","A1"]+[g[2] for g in GROUPS]].to_csv(weight_path,sep="\t",index=False,float_format="%.17g")
            if len(weights):
                sp=d/f"score_chr{chrom}"
                run_cmd([PLINK,"--pfile",str(pfiles[chrom]),"--keep",str(d/"test.keep.tsv"),"--read-freq",str(prefix)+".afreq",
                         "--score",str(weight_path),"1","2","header-read","cols=+scoresums","--score-col-nums","3-6",
                         "--threads",str(THREADS),"--memory",str(MEMORY_MB),"--out",str(sp)],sp)
                sc=pd.read_csv(Path(str(sp)+".sscore"),sep=r"\s+",dtype={"#IID":str,"IID":str}).rename(columns={"#IID":"IID"}).set_index("IID")
                require(sc.index.is_unique and set(sc.index)==set(test),"Score IDs differ from validation set")
                for _,_,score_col in GROUPS:
                    col=score_col+"_SUM";require(col in sc,f"Missing SUM column {col}")
                    score_total[score_col]+=pd.to_numeric(sc.loc[test,col],errors="raise").to_numpy()
            hp=d/f"heldout_chr{chrom}"
            q=cv.shape[1]
            run_cmd(base+["--keep",str(d/"test.keep.tsv"),"--glm","interaction","--parameters",f"1-{q+2}","--tests",str(q+2),"--out",str(hp)],hp)
            z=read_result(Path(str(hp)+f".{trait}.glm.linear"),unique=False)
            heldout_parts.append(z.loc[z.TEST.eq("ADDxAGE10")].copy())
        tr=pd.concat(train_parts,ignore_index=True)
        ht=pd.concat(heldout_parts,ignore_index=True)
        require(not tr.duplicated(KEY).any() and not ht.duplicated(KEY).any(),"Duplicate split GLM keys")
        tr.to_csv(d/"training_pooled_main.tsv",sep="\t",index=False);ht.to_csv(d/"heldout_1df_interaction.tsv",sep="\t",index=False)
        score_total.to_csv(d/"validation_panel_PGS.tsv",sep="\t",index_label="#IID")
        membership=panel.merge(tr[KEY+["P","ERRCODE","BETA","SE"]],on=KEY,how="left",validate="one_to_one")
        good=membership.ERRCODE.eq(".")&valid_p(membership.P)&np.isfinite(membership.BETA)&membership.SE.gt(0)
        for label,cut,score_col in GROUPS:membership[score_col]=good&(pd.Series(True,index=membership.index) if cut is None else membership.P.lt(cut))
        membership.to_csv(d/"training_defined_sets.tsv",sep="\t",index=False)
        y=ph.loc[test,trait].to_numpy();Ctest=cv.loc[test].to_numpy();age=cv.loc[test,"AGE10"].to_numpy()
        for label,cut,score_col in GROUPS:
            ids=membership.loc[membership[score_col],KEY]
            nset=len(ids)
            r=ols_pgs_hc3(y,Ctest,score_total[score_col].to_numpy(),age) if nset else {"P":np.nan,"status":"empty_training_set","N":len(y)}
            holdout_rows.append({"trait":trait,"group":label,"method":"panel_PGSxage_HC3","N_train_selected_SNP":nset,**r})
            h=ids.merge(ht[KEY+["P","ERRCODE"]],on=KEY,how="left",validate="one_to_one")
            pv=h.loc[h.ERRCODE.eq(".")&valid_p(h.P),"P"].to_numpy()
            r=acat(pv)
            holdout_rows.append({"trait":trait,"group":label,"method":"heldout_single_SNP_ACAT_classicOLS","N_train_selected_SNP":nset,
                                 "N_valid_P":len(pv),"P":r["P_ACAT"],"status":r["status"],"N_test":len(y)})
        # Export and process one fixed genotype block at a time; never load the full 1902-column matrix.
        for block,part in membership.groupby("block",sort=False):
            print(f"{trait}: variance-component block {block}",flush=True)
            chrom=int(part["#CHROM"].iloc[0]);prefix=d/"vc_blocks"/str(block);prefix.parent.mkdir(exist_ok=True)
            extract=Path(str(prefix)+".extract.txt");allele=Path(str(prefix)+".ALT.tsv")
            part.ID.to_csv(extract,index=False,header=False);part[["ID","ALT"]].to_csv(allele,sep="\t",index=False,header=False)
            run_cmd([PLINK,"--pfile",str(pfiles[chrom]),"--keep",str(d/"test.keep.tsv"),"--extract",str(extract),
                     "--export","A","--export-allele",str(allele),"--threads",str(THREADS),"--memory",str(MEMORY_MB),"--out",str(prefix)],prefix)
            raw=pd.read_csv(Path(str(prefix)+".raw"),sep=r"\s+",dtype={"IID":str}).set_index("IID")
            require(raw.index.is_unique and set(raw.index)==set(test),"Kernel export sample mismatch")
            for label,cut,score_col in GROUPS:
                selected=part.loc[part[score_col]]
                if not len(selected):continue
                af=train_freq[chrom].loc[selected.ID,"ALT_FREQS"].to_numpy(dtype=float)
                cols=[f"{v.ID}_{v.ALT}" for v in selected.itertuples(index=False)]
                require(set(cols).issubset(raw),"ALT dosage export orientation could not be checked")
                A=raw.loc[test,cols].apply(pd.to_numeric,errors="coerce").to_numpy(dtype=float)
                train_mac=2*len(train)*np.minimum(af,1-af)
                n_observed=np.isfinite(A).sum(axis=0)
                alt_count=np.nansum(A,axis=0)
                test_mac=np.minimum(alt_count,2*n_observed-alt_count)
                freq_ok=np.isfinite(af)&(af>0)&(af<1)&(train_mac>=VC_MIN_TRAIN_MAC)&(test_mac>=VC_MIN_TEST_MAC)
                A=A[:,freq_ok];af=af[freq_ok]
                if A.shape[1]==0:continue
                A=np.where(np.isfinite(A),A,2*af[None,:])
                A=(A-2*af[None,:])/np.sqrt(2*af*(1-af))[None,:]
                seed=VC_SEED^int(hashlib.sha256(f"{trait}|{block}|{score_col}".encode()).hexdigest()[:8],16)
                r=kernel_score(y,Ctest,A,age,seed)
                block_rows.append({"trait":trait,"group":label,"block":block,"N_train_selected_SNP":len(selected),**r})
            del raw;gc.collect()
            pd.DataFrame(block_rows).to_csv(OUT/"B_holdout/VC_block_results.tsv",sep="\t",index=False)
        del score_total,ph,cv,Ctest;gc.collect()
    blocks=pd.DataFrame(block_rows,columns=list(dict.fromkeys(["trait","group","block","status","P","N_SNP","MC_at_floor"]+[k for r in block_rows for k in r])))
    for trait in TRAITS:
        for label,_,_ in GROUPS:
            b=blocks.loc[blocks.trait.eq(trait)&blocks.group.eq(label)&blocks.status.eq("ok")]
            r=acat(b.P.to_numpy(),b.N_SNP.to_numpy()) if len(b) else acat([])
            holdout_rows.append({"trait":trait,"group":label,"method":"block_VC_ACAT_HC3score",
                                 "P":r["P_ACAT"],"status":r["status"],"N_blocks":len(b),
                                 "N_effective_SNP":int(b.N_SNP.sum()) if len(b) else 0,
                                 "N_MC_floor_blocks":int(b.MC_at_floor.sum()) if len(b) else 0})
    holdout=pd.DataFrame(holdout_rows)
    require(len(holdout)==PRIMARY_FAMILY,"Unexpected holdout family size")
    holdout["P_Bonf36"]=np.minimum(holdout.P*PRIMARY_FAMILY,1)
    holdout.to_csv(OUT/"B_holdout/PGS_VC_ACAT_holdout_summary.tsv",sep="\t",index=False)
    pd.DataFrame(split_rows).to_csv(OUT/"B_holdout/train_test_sample_counts.tsv",sep="\t",index=False)
    blocks.to_csv(OUT/"B_holdout/VC_block_results.tsv",sep="\t",index=False)
    display(pd.DataFrame(split_rows));display(holdout)
else:
    print("Only A existing-result ACAT was requested; training/validation skipped.")

## 解释与输出

- **A_existing_ACAT/existing_1df_4df_ACAT.tsv**：已有全样本交互结果的 ACAT，保留历史 4-df 原始 P。主效应筛选来自原全样本分析，属于原数据探索。
- **B_holdout/PGS_VC_ACAT_holdout_summary.tsv**：新训练/验证三方法汇总；分组和权重来自训练集，验证集只用于检验。实际 N 和 SNP 数会与原全样本分组不同。
- **B_holdout/VC_block_results.tsv**：kernel null rank、SNP 数、Q、HC3 score P、Gaussian 敏感性 P、MC 分辨率标记。块 P 不作单块发现声称。
- **B_holdout/{trait}/**：权重、score sums、训练主效应与验证交互结果、固定块 genotype 中间文件及全部命令；个体级文件只留 workspace，不需要传出。

PGS 显著更支持沿训练权重方向的聚合交互；VC/block-ACAT 显著可来自正负混合或稀疏信号；单 SNP ACAT 显著可能由少数小 P 驱动。不同结果不等于互相矛盾。它们都是关联证据，不直接证明纵向衰老因果、全部主效应模型正确或交互解释方差有多大。

模型、亲缘相关、协变量×age 的遗漏、数据选择和误差结构仍需检查。训练/验证分割不自动解决跨集亲缘；HC3 也不解决个体之间相关。ACAT 相关尾部近似不修复单 SNP P 校准问题。

**参考与验证范围：** [原始 SKAT variance-component score](https://pmc.ncbi.nlm.nih.gov/articles/PMC3135811/)、[ACAT](https://pmc.ncbi.nlm.nih.gov/articles/PMC6407498/)、[PLINK score SUM/mean-imputation](https://www.cog-genomics.org/plink/2.0/score)。实现中的 HC3 score multiplier、固定分块和合并策略是本次明确给出的分析选择，不声称等同于已发表软件的全局 G×E mixed model。交付无真实数据输出；通过合成数据、独立回归、数值边界和 kernel 校准检查后交付，真实结果需在 Workbench 执行。

In [ ]:
pd.DataFrame(sources).to_csv(OUT/"qc/source_files.tsv",sep="\t",index=False)
(OUT/"ANALYSIS_COMPLETE.json").write_text(json.dumps({"status":"complete","linear_run":str(LINEAR_RUN),
 "created_utc":datetime.now(timezone.utc).isoformat(),"train_test_requested":RUN_TRAIN_TEST,"split_seed":SPLIT_SEED,
 "candidate_panel_N":M,"score_scope":"existing_candidate_panel_not_genomewide_PRS",
 "VC_strategy":"fixed_block_kernel_score_then_ACAT_not_single_global_GRM","VC_block_size":VC_BLOCK_SNPS,
 "VC_null_MC":VC_N_SIM,"VC_min_train_MAC":VC_MIN_TRAIN_MAC,"VC_min_test_MAC":VC_MIN_TEST_MAC,"existing_ACAT_family":24,"holdout_primary_family":PRIMARY_FAMILY},indent=2)+"\n")
print("Finished:",OUT)